Avaliando a influência do El nino e da La nina na ocorrência de eventos extremos de precipitação no Brasil

In [36]:
import xarray as xr

# Carrega o arquivo mastigado instantaneamente
caminho_input = "/content/drive/MyDrive/DOUTORADO/DOUTORADO/br_maximas_anuais_oni_1961_2025.nc"
ds_maximas_anuais = xr.open_dataset(caminho_input)

print("Cubo de dados pronto para a modelagem estatística!")
print(ds_maximas_anuais)

Cubo de dados pronto para a modelagem estatística!
<xarray.Dataset> Size: 80MB
Dimensions:    (ano: 65, latitude: 393, longitude: 391)
Coordinates:
  * ano        (ano) int64 520B 1961 1962 1963 1964 1965 ... 2022 2023 2024 2025
  * latitude   (latitude) float32 2kB -33.85 -33.75 -33.65 ... 5.15 5.25 5.35
  * longitude  (longitude) float32 2kB -73.85 -73.75 -73.65 ... -34.95 -34.85
Data variables:
    pr         (ano, latitude, longitude) float64 80MB ...


In [27]:
# 1. Instalação silenciosa das bibliotecas necessárias
!pip install netcdf4 xarray pymannkendall statsmodels -q

# 2. Importações de praxe
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
from scipy.stats import genextreme, chi2

print("Ambiente preparado com sucesso!")

Ambiente preparado com sucesso!


In [28]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Aqui foi feita a busca da base de dados de El nino e La nina

In [29]:
import pandas as pd
import numpy as np

print("Buscando e processando o índice ONI da NOAA...")

url_oni = "https://psl.noaa.gov/data/correlation/oni.data"

# CORREÇÃO: Usando read_csv com sep=r'\s+' para forçar a divisão por espaços em branco
df_oni_bruto = pd.read_csv(url_oni, sep=r'\s+', skiprows=1, skipfooter=8, header=None, engine='python')

# Agora temos certeza absoluta de que há 13 colunas
meses = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']
df_oni_bruto.columns = ['ano'] + meses

# Substitui o valor nulo do arquivo da NOAA (-99.90) pelo NaN padrão do Python
df_oni_bruto = df_oni_bruto.replace(-99.90, np.nan)

# O pulo do gato: Desloca o mês de Dezembro para casar com o Verão do ano seguinte
df_oni_bruto['Dez_anterior'] = df_oni_bruto['Dez'].shift(1)
df_oni_bruto['ONI_Verao'] = df_oni_bruto[['Dez_anterior', 'Jan', 'Fev']].mean(axis=1)

# Cria a tabela final de covariável
df_covariavel_enso = df_oni_bruto[['ano', 'ONI_Verao']].dropna().copy()

# Garantindo que a coluna de ano seja um número inteiro para facilitar o cruzamento depois
df_covariavel_enso['ano'] = df_covariavel_enso['ano'].astype(int)

print(f"Série do El Niño calibrada de {df_covariavel_enso['ano'].min()} a {df_covariavel_enso['ano'].max()}!")

# ==============================================================================
# CHECKPOINT: SALVANDO O ÍNDICE ONI NO GOOGLE DRIVE
# ==============================================================================
# Aponta para a sua pasta oficial da tese
caminho_pasta = "/content/drive/MyDrive/DOUTORADO/DOUTORADO/"
caminho_arquivo_oni = caminho_pasta + "indice_oni_verao.csv"

# Salva a tabela limpa e sem o índice automático do Pandas
df_covariavel_enso.to_csv(caminho_arquivo_oni, index=False)

print(f"✅ Sucesso absoluto! Arquivo salvo em: {caminho_arquivo_oni}")

Buscando e processando o índice ONI da NOAA...
Série do El Niño calibrada de 1950 a 2026!
✅ Sucesso absoluto! Arquivo salvo em: /content/drive/MyDrive/DOUTORADO/DOUTORADO/indice_oni_verao.csv


Agora vou selecionar as máximas diarias anuais da série histórica de Xavier

In [30]:
import xarray as xr

# Carrega o arquivo mastigado instantaneamente
caminho_input = "/content/drive/MyDrive/DOUTORADO/DOUTORADO/br_maximas_anuais_oni_1961_2025.nc"
ds_maximas_anuais = xr.open_dataset(caminho_input)

print("Cubo de dados pronto para a modelagem estatística!")
print(ds_maximas_anuais)

Cubo de dados pronto para a modelagem estatística!
<xarray.Dataset> Size: 80MB
Dimensions:    (ano: 65, latitude: 393, longitude: 391)
Coordinates:
  * ano        (ano) int64 520B 1961 1962 1963 1964 1965 ... 2022 2023 2024 2025
  * latitude   (latitude) float32 2kB -33.85 -33.75 -33.65 ... 5.15 5.25 5.35
  * longitude  (longitude) float32 2kB -73.85 -73.75 -73.65 ... -34.95 -34.85
Data variables:
    pr         (ano, latitude, longitude) float64 80MB ...


Agora vou processar as GEVS para o estado de MG

In [31]:
# ==============================================================================
# REPROCESSAMENTO GEV (MINAS GERAIS) COM SINCRONIZAÇÃO DO ONI E CHUVA
# ==============================================================================
import xarray as xr
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

# Caminho base unificado para a sua pasta da tese
caminho_pasta = "/content/drive/MyDrive/DOUTORADO/DOUTORADO/"

print("1. Carregando a matéria-prima do Drive...")
caminho_input = caminho_pasta + "br_maximas_anuais_oni_1961_2025.nc"
ds_br = xr.open_dataset(caminho_input)

print("2. Recortando o Brasil para focar apenas em Minas Gerais...")
lat_min, lat_max = -23.0, -14.0
lon_min, lon_max = -51.5, -39.5

try:
    ds_mg = ds_br.sel(latitude=slice(lat_max, lat_min), longitude=slice(lon_min, lon_max))
    if len(ds_mg.latitude) == 0:
        ds_mg = ds_br.sel(latitude=slice(lat_min, lat_max), longitude=slice(lon_min, lon_max))
except:
    ds_mg = ds_br.sel(latitude=slice(lat_min, lat_max), longitude=slice(lon_min, lon_max))

print(f" -> Malha de MG separada: {len(ds_mg.latitude)} lats x {len(ds_mg.longitude)} lons.")

# ==============================================================================
# CHECKPOINT 1: SALVANDO O RECORTE REGIONAL NO DRIVE
# ==============================================================================
print("\n[CHECKPOINT 1] Salvando o recorte de dados de Minas Gerais no Drive...")
caminho_recorte = caminho_pasta + "mg_maximas_anuais.nc"
ds_mg.to_netcdf(caminho_recorte)
print(f"✅ Recorte regional salvo com sucesso em: {caminho_recorte}")

# ==============================================================================
# SINCRONIZANDO O ÍNDICE ONI COM OS ANOS DA CHUVA
# ==============================================================================
print("\n3. Resgatando o ONI salvo e sincronizando os anos com o NetCDF...")
df_oni = pd.read_csv(caminho_pasta + "indice_oni_verao.csv")

# Extrai os anos que existem no arquivo NetCDF
anos_chuva = ds_mg['ano'].values

# Filtra a tabela do ONI para pegar exatamente os anos correspondentes e garante a ordem
df_oni_filtrado = df_oni[df_oni['ano'].isin(anos_chuva)].sort_values('ano')
vetor_oni = df_oni_filtrado['ONI_Verao'].values

print(f" -> Sincronismo ok: {len(vetor_oni)} valores de ONI para {len(anos_chuva)} anos de chuva.")

# ==============================================================================
# MOTOR GEV
# ==============================================================================
# Preparando a matriz em branco para guardar os resultados
lats = ds_mg.latitude.values
lons = ds_mg.longitude.values
matriz_resultados = np.full((len(lats), len(lons)), np.nan)

print("\n4. Ligando o Motor GEV (Vai demorar um bocado, hora do café!)...")
total_pixels = len(lats) * len(lons)
barra_progresso = tqdm(total=total_pixels, desc="Processando Pixels de MG")

for i, lat in enumerate(lats):
    for j, lon in enumerate(lons):
        # CORREÇÃO: Usando a variável 'pr' conforme o seu banco de dados
        serie_pixel = ds_mg['pr'].values[:, i, j]

        # Só calcula se o pixel não for "água" ou sem dados (NaN)
        if not np.all(np.isnan(serie_pixel)):
            res = ajustar_gev_4_modelos(serie_pixel, vetor_oni)
            id_vencedor = res[6] # O ID do modelo vencedor
            matriz_resultados[i, j] = id_vencedor

        barra_progresso.update(1)

barra_progresso.close()

# ==============================================================================
# CHECKPOINT 2: SALVANDO OS RESULTADOS
# ==============================================================================
print("\n5. Empacotando e salvando a matriz de resultados no seu Drive...")
da_resultado_mg = xr.DataArray(
    matriz_resultados,
    coords={'latitude': lats, 'longitude': lons},
    dims=['latitude', 'longitude'],
    name='modelo_vencedor'
)

caminho_output = caminho_pasta + "resultado_gev_mg.nc"
da_resultado_mg.to_netcdf(caminho_output)

print(f"\n✅ SUCESSO ABSOLUTO! A matriz final está salva e blindada.")
print(f" -> Arquivo: {caminho_output}")

1. Carregando a matéria-prima do Drive...
2. Recortando o Brasil para focar apenas em Minas Gerais...
 -> Malha de MG separada: 90 lats x 120 lons.

[CHECKPOINT 1] Salvando o recorte de dados de Minas Gerais no Drive...


PermissionError: [Errno 13] Permission denied: '/content/drive/MyDrive/DOUTORADO/DOUTORADO/mg_maximas_anuais.nc'

In [32]:
# ==============================================================================
# CÉLULA 8: PROCESSAMENTO ESPACIAL E EXPORTAÇÃO DOS RESULTADOS (MG)
# ==============================================================================
import xarray as xr
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

caminho_pasta = "/content/drive/MyDrive/DOUTORADO/DOUTORADO/"

print("1. Carregando os Checkpoints salvos no Drive...")
# Carrega o recorte espacial já mastigado
ds_mg = xr.open_dataset(caminho_pasta + "mg_maximas_anuais.nc")

# Carrega o índice ONI limpo
df_oni = pd.read_csv(caminho_pasta + "indice_oni_verao.csv")

print("2. Sincronizando a linha do tempo (Chuva x El Niño)...")
anos_chuva = ds_mg['ano'].values
df_oni_filtrado = df_oni[df_oni['ano'].isin(anos_chuva)].sort_values('ano')
vetor_oni = df_oni_filtrado['ONI_Verao'].values

print(f" -> Tudo alinhado: {len(vetor_oni)} anos de El Niño para {len(anos_chuva)} anos de chuva.")

# 3. Preparando a matriz em branco para guardar os resultados
lats = ds_mg.latitude.values
lons = ds_mg.longitude.values
matriz_resultados = np.full((len(lats), len(lons)), np.nan)

print("\n3. Acionando o Motor GEV (Isso vai exigir um pouco da máquina!)...")
total_pixels = len(lats) * len(lons)
barra_progresso = tqdm(total=total_pixels, desc="Varrendo MG pixel a pixel")

for i, lat in enumerate(lats):
    for j, lon in enumerate(lons):
        # Chama a variável 'pr' (precipitação) corretamente
        serie_pixel = ds_mg['pr'].values[:, i, j]

        # Evita processar pixels vazios (ex: oceano ou fora do contorno)
        if not np.all(np.isnan(serie_pixel)):
            # Aciona o motor da célula anterior
            res = ajustar_gev_4_modelos(serie_pixel, vetor_oni)
            id_vencedor = res[6] # O ID do modelo selecionado pelo AIC

            matriz_resultados[i, j] = id_vencedor

        barra_progresso.update(1)

barra_progresso.close()

# 4. Salvando o arquivo de ouro
print("\n4. Empacotando e gravando a Matriz de Modelos no Google Drive...")
da_resultado_mg = xr.DataArray(
    matriz_resultados,
    coords={'latitude': lats, 'longitude': lons},
    dims=['latitude', 'longitude'],
    name='modelo_vencedor'
)

caminho_output = caminho_pasta + "resultado_gev_mg.nc"
da_resultado_mg.to_netcdf(caminho_output)

print(f"\n✅ SUCESSO ABSOLUTO! O processamento bruto terminou.")
print(f"O arquivo definitivo está blindado e salvo em: {caminho_output}")

1. Carregando os Checkpoints salvos no Drive...


ValueError: did not find a match in any of xarray's currently installed IO backends ['h5netcdf', 'scipy']. Consider explicitly selecting one of the installed engines via the ``engine`` parameter, or installing additional IO dependencies, see:
https://docs.xarray.dev/en/stable/getting-started-guide/installing.html
https://docs.xarray.dev/en/stable/user-guide/io.html

In [34]:
# ==============================================================================
# CÉLULA DEFINITIVA: PROCESSAMENTO COMPLETO DE MINAS GERAIS E PLOTAGEM
# ==============================================================================
import numpy as np
import pandas as pd
import xarray as xr
import warnings
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, BoundaryNorm
from tqdm.notebook import tqdm

caminho_pasta = "/content/drive/MyDrive/DOUTORADO/DOUTORADO/"
warnings.filterwarnings('ignore')

print("1. Carregando os dados de Minas Gerais e o Índice ONI...")
ds_mg = xr.open_dataset(caminho_pasta + "mg_maximas_anuais.nc")
df_oni = pd.read_csv(caminho_pasta + "indice_oni_verao.csv")

# ==============================================================================
# O PULO DO GATO: Alinhamento perfeito dos 65 anos
# ==============================================================================
anos_chuva = ds_mg['ano'].values
anos_oni = df_oni['ano'].values
anos_comuns = np.intersect1d(anos_chuva, anos_oni)

# Filtra ambas as bases para terem EXATAMENTE a mesma linha do tempo
ds_mg_sincronizado = ds_mg.sel(ano=anos_comuns)
df_enso_alinhado = df_oni[df_oni['ano'].isin(anos_comuns)].sort_values('ano')

lats = ds_mg_sincronizado.latitude.values
lons = ds_mg_sincronizado.longitude.values
vetor_oni = df_enso_alinhado['ONI_Verao'].values

print(f" -> Sincronia perfeita: {len(anos_comuns)} anos alinhados para o Motor GEV.")
print(f" -> Malha do estado: {len(lats)} Lats x {len(lons)} Lons.")

matriz_vencedores = np.full((len(lats), len(lons)), np.nan)

print("\n2. Ligando o Motor GEV para o Estado Inteiro (Paciência, isso vai exigir da máquina!)...")
# Varredura completa
for i in tqdm(range(len(lats)), desc="Processando Latitudes (MG)"):
    for j in range(len(lons)):
        serie_pixel = ds_mg_sincronizado['pr'].isel(latitude=i, longitude=j).values

        # Só calcula se houver dados de chuva válidos naquele pixel
        if not np.isnan(serie_pixel).all():
            resultado = ajustar_gev_4_modelos(serie_pixel, vetor_oni)
            matriz_vencedores[i, j] = resultado[6]

pixels_validos = np.count_nonzero(~np.isnan(matriz_vencedores))
print(f"\n[DIAGNÓSTICO FINAL] Sucesso! Modelos válidos calculados para {pixels_validos} pixels!")

# ==============================================================================
# SALVAMENTO BLINDADO NO DRIVE
# ==============================================================================
print("\n3. Empacotando e gravando a Matriz de Ouro no Google Drive...")
da_resultado_mg = xr.DataArray(
    matriz_resultados,
    coords={'latitude': lats, 'longitude': lons},
    dims=['latitude', 'longitude'],
    name='modelo_vencedor'
)

caminho_output = caminho_pasta + "resultado_gev_mg.nc"
da_resultado_mg.to_netcdf(caminho_output)
print(f"✅ Arquivo definitivo salvo em: {caminho_output}")

# ==============================================================================
# DESENHANDO O MAPA DO ESTADO
# ==============================================================================
print("\n4. Renderizando o mapa climatológico na tela...")

cores_hex = ['#005b96', '#d62728', '#ff7f0e', '#9467bd']
cmap_discreto = ListedColormap(cores_hex)
limites = [-0.5, 0.5, 1.5, 2.5, 3.5]
norma = BoundaryNorm(limites, cmap_discreto.N)

fig, ax = plt.subplots(figsize=(12, 12))

X, Y = np.meshgrid(da_resultado_mg.longitude, da_resultado_mg.latitude)
ax.pcolormesh(X, Y, da_resultado_mg.values, cmap=cmap_discreto, norm=norma, shading='auto', edgecolor='white', linewidth=0.1)

nomes_modelos = [
    'Modelo 0: Estacionário (Sem efeito do El Niño)',
    'Modelo 1: Efeito apenas na Média (Localização)',
    'Modelo 2: Efeito apenas na Variabilidade (Escala)',
    'Modelo 3: Efeito na Média e na Variabilidade'
]

legend_handles = [mpatches.Patch(color=cores_hex[i], label=nomes_modelos[i]) for i in range(4)]
ax.legend(handles=legend_handles, title="Modelo Vencedor (Critério AIC)", loc='lower left',
          bbox_to_anchor=(0.02, 0.05), frameon=True, facecolor='white', framealpha=0.95, fontsize=10)

ax.set_title("Impacto do El Niño nos Extremos de Precipitação Diária\nEstado de Minas Gerais",
             fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel("Longitude", fontsize=12)
ax.set_ylabel("Latitude", fontsize=12)

plt.tight_layout()
plt.show()

1. Carregando os dados de Minas Gerais e o Índice ONI...


ValueError: did not find a match in any of xarray's currently installed IO backends ['h5netcdf', 'scipy']. Consider explicitly selecting one of the installed engines via the ``engine`` parameter, or installing additional IO dependencies, see:
https://docs.xarray.dev/en/stable/getting-started-guide/installing.html
https://docs.xarray.dev/en/stable/user-guide/io.html

In [24]:
# ==============================================================================
# RAIO-X DO FICHEIRO NETCDF (VERIFICANDO A EXISTÊNCIA DE CHUVA)
# ==============================================================================
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

caminho_pasta = "/content/drive/MyDrive/DOUTORADO/DOUTORADO/"
ds_mg = xr.open_dataset(caminho_pasta + "mg_maximas_anuais.nc")

print("--- DIAGNÓSTICO DO FICHEIRO BRUTO ---")
total_dados = ds_mg['pr'].size
nans = int(np.isnan(ds_mg['pr'].values).sum())
validos = total_dados - nans

print(f"-> Total de registos na malha: {total_dados}")
print(f"-> Registos de chuva VÁLIDOS: {validos}")
print(f"-> Registos NULOS (NaN): {nans}")

if validos > 0:
    print("\nÓtimo! Temos dados reais. A desenhar o mapa bruto do primeiro ano...")
    fig, ax = plt.subplots(figsize=(10, 8))
    ds_mg['pr'].isel(ano=0).plot(ax=ax, cmap='Blues')
    ax.set_title(f"Precipitação Máxima Bruta - Ano {ds_mg['ano'].values[0]}")
    plt.show()
else:
    print("\n⚠️ ALERTA VERMELHO: O ficheiro está 100% VAZIO (apenas contém NaN).")
    print("A variável 'pr' não tem dados numéricos. É necessário recarregar o ficheiro NetCDF original de precipitação!")

--- DIAGNÓSTICO DO FICHEIRO BRUTO ---
-> Total de registos na malha: 702000
-> Registos de chuva VÁLIDOS: 0
-> Registos NULOS (NaN): 702000

⚠️ ALERTA VERMELHO: O ficheiro está 100% VAZIO (apenas contém NaN).
A variável 'pr' não tem dados numéricos. É necessário recarregar o ficheiro NetCDF original de precipitação!


In [25]:
# ==============================================================================
# RESGATE DA MATÉRIA-PRIMA: RECRIANDO O RECORTE DE MINAS GERAIS À PROVA DE BALAS
# ==============================================================================
import xarray as xr
import numpy as np

caminho_pasta = "/content/drive/MyDrive/DOUTORADO/DOUTORADO/"
caminho_br = caminho_pasta + "br_maximas_anuais_oni_1961_2025.nc"

print("1. Abrindo o arquivo Mestre do Brasil...")
ds_br = xr.open_dataset(caminho_br)

# Verifica se a matéria-prima original está íntegra
validos_br = int((~np.isnan(ds_br['pr'].values)).sum())
print(f"-> Dados VÁLIDOS no arquivo do Brasil inteiro: {validos_br}")

if validos_br == 0:
    print("\n❌ Eita! O arquivo original do Brasil também perdeu os dados (só tem NaN).")
    print("Nesse caso, nós vamos precisar voltar e rodar o script que constrói o 'br_maximas_anuais' do zero.")
else:
    print("\n2. Os dados estão a salvo! Refazendo o recorte de Minas Gerais...")

    # Ajusta a longitude automaticamente caso o arquivo use o formato 0 a 360
    lon_min, lon_max = -51.5, -39.5
    if float(ds_br.longitude.max()) > 180:
        lon_min += 360
        lon_max += 360

    # O comando .where() filtra geograficamente sem se importar com a ordem do NetCDF
    ds_mg_novo = ds_br.where(
        (ds_br.latitude >= -23.0) & (ds_br.latitude <= -14.0) &
        (ds_br.longitude >= lon_min) & (ds_br.longitude <= lon_max),
        drop=True
    )

    validos_mg = int((~np.isnan(ds_mg_novo['pr'].values)).sum())
    print(f"-> Malha recortada: {len(ds_mg_novo.latitude)} lats x {len(ds_mg_novo.longitude)} lons.")
    print(f"-> Registros VÁLIDOS na caixa de Minas Gerais: {validos_mg}")

    if validos_mg > 0:
        caminho_novo_mg = caminho_pasta + "mg_maximas_anuais.nc"
        ds_mg_novo.to_netcdf(caminho_novo_mg)
        print(f"\n✅ SUCESSO ABSOLUTO! O arquivo de Minas foi curado e salvo em: {caminho_novo_mg}")
        print("Pode voltar lá na nossa 'CÉLULA DEFINITIVA' e dar o play no Motor GEV novamente!")
    else:
        print("\n⚠️ O recorte caiu em uma área sem estações pluviométricas. Confirme as coordenadas do seu NetCDF!")

1. Abrindo o arquivo Mestre do Brasil...
-> Dados VÁLIDOS no arquivo do Brasil inteiro: 0

❌ Eita! O arquivo original do Brasil também perdeu os dados (só tem NaN).
Nesse caso, nós vamos precisar voltar e rodar o script que constrói o 'br_maximas_anuais' do zero.
